# Lab 19-10 · A Timing Engine and a Mapper on Real Liberty Data

**Predict first**, then run. The handout is [README.md](README.md). The data are SKY130 `sky130_fd_sc_hd` tables (Apache License 2.0), typical corner unless stated.

In [1]:
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sta
import techmap as tm
from techmap import IN, INV, NAND

lib = sta.load_library()
print(sorted(lib["cells"]))

['a21oi_1', 'a22oi_1', 'and2_1', 'buf_1', 'dfxtp_1', 'inv_1', 'nand2_1', 'nand3_1', 'nand4_1', 'nor2_1', 'o21ai_1']


## Part 1 · Tables, interpolation and extrapolation

*Predict* the rising delay of `inv_1` at 0.25 pF (input slew 0.0531329 ns), beyond the table's last load (0.181 pF).

In [2]:
arc = lib["cells"]["inv_1"]["arcs"]["A"]
for slew, row in zip(arc["index_1"], arc["cell_rise"]):
    plt.loglog(np.array(arc["index_2"]) * 1e3, np.array(row) * 1e3, "o-", ms=3, label=f"{slew:.3g} ns")
plt.xlabel("load (fF)"); plt.ylabel("cell_rise (ps)"); plt.legend(fontsize=7); plt.show()
print("Example 1 point, rise/fall (ps):", [round(1e3 * sta.arc_delay(lib, "inv_1", "A", e, 0.08, 0.005)[0], 2) for e in ("rise", "fall")])
print("extrapolated to 0.25 pF (ns):", round(sta.arc_delay(lib, "inv_1", "A", "rise", 0.0531329, 0.25)[0], 4))
print("last two table points (ns):", arc["cell_rise"][2][5:], "at", arc["index_2"][5:], "pF")
ss = {"cells": lib["other_corners"]["ss_100C_1v60"]}
for e in ("rise", "fall"):
    r = sta.arc_delay(ss, "inv_1", "A", e, 0.08, 0.005)[0] / sta.arc_delay(lib, "inv_1", "A", e, 0.08, 0.005)[0]
    print(f"ss_100C_1v60 / tt, {e}: {r:.3f}")

Example 1 point, rise/fall (ps): [75.88, 53.38]
extrapolated to 0.25 pF (ns): 1.4513
last two table points (ns): [0.4194971, 1.0619717] at [0.0678883, 0.181284] pF
ss_100C_1v60 / tt, rise: 1.334
ss_100C_1v60 / tt, fall: 1.622


## Part 2 · Slew propagation down a chain

*Predict* how much later the output of four `inv_1` cells (driving 0.01 pF) arrives when the input slew is 0.5 ns instead of 0.01 ns, and which stages absorb the difference.

In [3]:
def chain(in_slew, n=4, load=0.01):
    nl = sta.Netlist(lib, {"x": (0.0, 0.0, in_slew)}, {"y": (f"n{n}", load)})
    prev = "x"
    for k in range(1, n + 1):
        nl.add(f"u{k}", "inv_1", {"A": prev}, f"n{k}")
        prev = f"n{k}"
    nl.analyze()
    return nl

for s in (0.01, 0.5):
    nl = chain(s)
    print(f"input slew {s} ns: output rise {1e3 * nl.at[('n4', 'rise')]:.2f} ps, fall {1e3 * nl.at[('n4', 'fall')]:.2f} ps")
    print("   slews per stage (rise, ps):", [round(1e3 * nl.slew[(f'n{k}', 'rise')], 1) for k in range(1, 5)])
    print("   worst path to the rising output:", [(n, e, round(1e3 * t, 1)) for n, e, t in nl.worst_path("n4", "rise")])

input slew 0.01 ns: output rise 160.42 ps, fall 147.01 ps
   slews per stage (rise, ps): [29.2, 29.2, 29.2, 91.6]
   worst path to the rising output: [('x', 'rise', 0.0), ('n1', 'fall', 20.8), ('n2', 'rise', 54.4), ('n3', 'fall', 81.8), ('n4', 'rise', 160.4)]
input slew 0.5 ns: output rise 244.35 ps, fall 282.08 ps
   slews per stage (rise, ps): [104.7, 38.2, 30.0, 91.6]
   worst path to the rising output: [('x', 'rise', 0.0), ('n1', 'fall', 75.5), ('n2', 'rise', 134.8), ('n3', 'fall', 165.1), ('n4', 'rise', 244.4)]


## Part 3 · Path explosion: graph-based against path-based

*Predict* the number of paths through 10 diamonds, and whether graph-based analysis gives away much here.

In [4]:
def diamonds(n, in_slew=0.05):
    nl = sta.Netlist(lib, {"x0": (0.0, 0.0, in_slew)}, {"y": (f"x{n}", 0.005)})
    for k in range(n):
        nl.add(f"a{k}", "inv_1", {"A": f"x{k}"}, f"a{k}")
        nl.add(f"b{k}", "buf_1", {"A": f"x{k}"}, f"b{k}")
        nl.add(f"g{k}", "nand2_1", {"A": f"a{k}", "B": f"b{k}"}, f"x{k + 1}")
    return nl

for n in (2, 6, 10):
    nl = diamonds(n)
    t0 = time.perf_counter(); at, _ = nl.analyze(); t_gba = time.perf_counter() - t0
    t0 = time.perf_counter(); pb = nl.path_based("x0", f"x{n}"); t_pba = time.perf_counter() - t0
    print(f"{n:2d} diamonds, {nl.count_paths('x0', f'x{n}'):5d} paths: GBA rise {1e3 * at[(f'x{n}', 'rise')]:.2f} ps "
          f"({1e3 * t_gba:.1f} ms), PBA rise {1e3 * pb['rise']:.2f} ps ({1e3 * t_pba:.1f} ms)")
nl = diamonds(20); t0 = time.perf_counter(); nl.analyze()
print("20 diamonds,", sta.count_paths(20), "paths, graph-based in", round(1e3 * (time.perf_counter() - t0), 1), "ms")

 2 diamonds,     4 paths: GBA rise 282.43 ps (0.2 ms), PBA rise 282.41 ps (0.1 ms)
 6 diamonds,    64 paths: GBA rise 844.81 ps (0.4 ms), PBA rise 844.78 ps (1.5 ms)
10 diamonds,  1024 paths: GBA rise 1407.19 ps (0.8 ms), PBA rise 1407.15 ps (21.2 ms)
20 diamonds, 1048576 paths, graph-based in 2.8 ms


## Part 4 · Mapping for area and for delay

*Predict* the cells chosen for $\\overline{ABCD + E}$ and for $\\overline{AB + CD}$.

In [5]:
cells = list(tm.PATTERNS)
area = {c: lib["cells"][c]["area"] for c in cells}
delay = {c: round(1e3 * max(sta.arc_delay(lib, c, p, e, 0.05, 0.005)[0]
                            for p in lib["cells"][c]["arcs"] for e in ("rise", "fall")), 1) for c in cells}
print("delays (ps):", delay)
A, B, C, D, E = map(IN, "ABCDE")
trees = {"!(ABCD + E)": INV(NAND(NAND(INV(NAND(A, B)), INV(NAND(C, D))), INV(E))),
         "!(AB + CD)": INV(NAND(NAND(A, B), NAND(C, D)))}
for name, t in trees.items():
    a, d = tm.min_area(t, area), tm.min_delay(t, delay)
    print(f"{name}: min area {a[0]:.4f} um^2 {a[1]} | min delay {d[0]:.1f} ps {d[1]}")

delays (ps): {'inv_1': 63.4, 'nand2_1': 77.1, 'and2_1': 163.0, 'nor2_1': 125.4, 'nand3_1': 92.8, 'nand4_1': 108.4, 'a21oi_1': 168.7, 'a22oi_1': 207.0, 'o21ai_1': 154.8}
!(ABCD + E): min area 16.2656 um^2 ['and2_1', 'nand4_1', 'inv_1'] | min delay 248.9 ps ['inv_1', 'nand2_1', 'nand4_1', 'inv_1']
!(AB + CD): min area 7.5072 um^2 ['a22oi_1'] | min delay 207.0 ps ['a22oi_1']


## Part 5 · Derates and CRPR

*Predict* the symmetric derate $d$ (late $1 + d$, early $1 - d$) at which Example 19-10.5's slack reaches zero, with and without CRPR.

In [6]:
ds = np.linspace(0, 0.35, 141)
for crpr, style in ((False, "--"), (True, "-")):
    plt.plot(100 * ds, [sta.ocv_setup_slack(2.0, 0.40, 0.20, 0.22, 0.30, 1.10, 0.10, 1 + d, 1 - d, crpr) for d in ds],
             style, label="with CRPR" if crpr else "without CRPR")
plt.axhline(0, color="k", lw=0.5); plt.xlabel("derate d (%)"); plt.ylabel("setup slack (ns)"); plt.legend(); plt.show()
from scipy.optimize import brentq
for crpr in (False, True):
    d0 = brentq(lambda d: sta.ocv_setup_slack(2.0, 0.40, 0.20, 0.22, 0.30, 1.10, 0.10, 1 + d, 1 - d, crpr), 0, 0.5)
    print("CRPR" if crpr else "no CRPR", f": zero slack at d = {100 * d0:.1f} %")

no CRPR : zero slack at d = 19.8 %
CRPR : zero slack at d = 28.6 %


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).